In [1]:


import pandas as pd

df = pd.read_csv("../data/processed/uganda_fake_news_master_dataset.csv")

print(df.shape)
print(df['label'].value_counts())
df.head()

(1884, 8)
label
TRUE    944
FAKE    940
Name: count, dtype: int64


,id,text,label,source,platform_type,language,date_collected,text_length
0,UG_TRUE_001,The Ministry of Health confirms no outbreak of...,TRUE,Daily Monitor,News Website,English,2025-01-10,11
1,UG_FAKE_001,Drinking hot water every 15 minutes kills COVI...,FAKE,AfricaCheck,Fact-Check,English,2025-01-10,12
2,UG_FAKE_002,Government has approved free electricity for a...,FAKE,PesaCheck,Fact-Check,English,2025-01-11,11
3,UG_TRUE_002,Parliament passes new amendment to the Nationa...,TRUE,New Vision,News Website,English,2025-01-11,10
4,UG_TRUE_003,Museveni alisema hakuna lockdown tena nchini U...,TRUE,BBC Africa,News Website,Mixed,2025-01-12,7


In [2]:

# Missing values
df.isnull().sum()

# Duplicate texts
df.duplicated(subset=["text"]).sum()

np.int64(14)

In [3]:
#drop duplicate texts
df = df.drop_duplicates(subset=["text"])

In [4]:
# 
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# Text preprocessing function

def preprocess_text(text):
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. Remove special characters but KEEP letters & numbers
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 4. Tokenize
    tokens = word_tokenize(text)
    
    # 5. Remove English stopwords only
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

In [6]:
# Apply preprocessing to the 'text' column and create a new 'clean_text' column
df['clean_text'] = df['text'].apply(preprocess_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,The Ministry of Health confirms no outbreak of...,ministry health confirms outbreak ebola kampala
1,Drinking hot water every 15 minutes kills COVI...,drinking hot water every 15 minutes kills covi...
2,Government has approved free electricity for a...,government approved free electricity ugandans ...
3,Parliament passes new amendment to the Nationa...,parliament passes new amendment national id re...
4,Museveni alisema hakuna lockdown tena nchini U...,museveni alisema hakuna lockdown tena nchini u...


In [7]:
# Encode labels: FAKE=0, TRUE=1
df['label_encoded'] = df['label'].map({'FAKE': 0, 'TRUE': 1})

df[['label', 'label_encoded']].head()

,label,label_encoded
0,TRUE,1
1,FAKE,0
2,FAKE,0
3,TRUE,1
4,TRUE,1


In [8]:

# Check for any nulls in the encoded labels
df['label_encoded'].isnull().sum()

np.int64(0)

In [9]:
# Display encoded labels
df[['label', 'label_encoded']].head()


,label,label_encoded
0,TRUE,1
1,FAKE,0
2,FAKE,0
3,TRUE,1
4,TRUE,1


In [10]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
# Check for overlap between training and testing sets
df = df.drop_duplicates(subset=["clean_text"])

In [12]:
# Save the preprocessed dataset
df.to_csv("../data/processed/uganda_fake_news_preprocessed_master_dataset.csv", index=False)